# Spike 03 — English OCR with Chandra 2

**Goal:** Extract structured English text from the Tranquil City page images using Chandra 2.

**Time box:** 1 hour

**Question to answer:** Does Chandra produce well-structured Markdown (especially tables) from these English report pages?

**Done when:** `.md` files in `ocr_output/english/` contain readable text with tables preserved as Markdown.

---

### Why Chandra 2 for English?

| Model | Overall Average (90 langs) | Arabic |
|---|---|---|
| **Chandra 2** | **72.7%** ✅ | 68.4% |
| Gemini 2.5 Flash | 60.8% | 84.4% |

Chandra wins on overall quality — and crucially, it outputs **structured HTML/Markdown** that  
preserves table cells, column headers, and multi-column layouts. This matters a lot for urban reports full of statistics.

### API Key Setup
1. Go to [chandra.datalab.to](https://chandra.datalab.to)
2. Sign up → get \$5 free credits
3. Copy API key → add to `.env`: `CHANDRA_API_KEY=your_key_here`

---
### ⚠️ Note on Chandra's Python Client
Chandra's SDK is relatively new. If the import or method names below don't match exactly,  
check their latest README at https://github.com/datalab-to/chandra and adjust accordingly.  
The logic and structure here is correct — only method names may differ.

## Step 1 — Check Chandra Installation

In [ ]:
# Check if chandra is installed — install if not
try:
    import chandra
    print(f"✅ Chandra installed: {chandra.__version__}")
except ImportError:
    print("Installing chandra...")
    import subprocess
    subprocess.run(["pip", "install", "chandra-ocr", "-q"])
    import chandra
    print(f"✅ Chandra installed: {chandra.__version__}")

## Step 2 — Setup

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os
import time

load_dotenv(dotenv_path="../.env")

CHANDRA_API_KEY = os.getenv("CHANDRA_API_KEY")
if not CHANDRA_API_KEY:
    raise ValueError("CHANDRA_API_KEY not found in .env file")

# Initialise Chandra client (adjust class name if needed per their docs)
client = chandra.Client(api_key=CHANDRA_API_KEY)

print("✅ Chandra client ready")

## Step 3 — OCR a Single Page First

In [ ]:
def ocr_page_english(image_path: str, output_format: str = "markdown") -> str:
    """
    Sends a page image to Chandra and returns structured Markdown.

    output_format options: 'markdown', 'html', 'json'
    Use 'markdown' for RAG pipelines.
    Use 'html'     if tables aren't rendering correctly in markdown.
    """
    result = client.ocr(
        image          = image_path,
        output_format  = output_format,
        preserve_layout= True
    )
    # Adjust attribute name based on actual Chandra response object
    return result.text if hasattr(result, 'text') else result.get('text', str(result))


en_images = sorted(Path("../data/images_en").glob("*.png"))

if not en_images:
    print("❌ No English images found — run Spike 01 first")
else:
    print(f"Found {len(en_images)} English page images")
    print(f"Testing OCR on: {en_images[0].name}")
    print()

    test_result = ocr_page_english(str(en_images[0]))

    print("=== CHANDRA OCR OUTPUT — Page 1 ===")
    print(test_result[:1500])
    print()
    print(f"Total characters: {len(test_result)}")

## Step 4 — Compare Formats on a Table Page

Urban reports are full of tables. Let's see which format preserves them best.

In [ ]:
# Try a page that likely has a table (pick page 4 or 5)
TABLE_PAGE_IDX = 3   # 0-indexed, so this is page 4

if len(en_images) > TABLE_PAGE_IDX:
    table_page = en_images[TABLE_PAGE_IDX]
    print(f"Testing on: {table_page.name}")

    print("\n=== OUTPUT FORMAT: markdown ===")
    md_result = ocr_page_english(str(table_page), output_format="markdown")
    print(md_result[:1000])

    print("\n=== OUTPUT FORMAT: html ===")
    html_result = ocr_page_english(str(table_page), output_format="html")
    print(html_result[:1000])

    # Decision: whichever preserves tables better, use that
    print("\n=== COMPARISON ===")
    print(f"Markdown has table markers (|): {'✅ Yes' if '|' in md_result else '❌ No'}")
    print(f"HTML has table tags (<table>) : {'✅ Yes' if '<table' in html_result.lower() else '❌ No'}")

## Step 5 — Run OCR on All English Pages

In [ ]:
OUTPUT_DIR = Path("../ocr_output/english")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to whichever format worked better in Step 4
BEST_FORMAT = "markdown"   # change to 'html' if tables looked better in html

results = {}

for i, img_path in enumerate(en_images):
    out_path = OUTPUT_DIR / img_path.with_suffix(".md").name

    if out_path.exists():
        print(f"⏭  Skipping {img_path.name} — already processed")
        results[img_path.stem] = out_path.read_text(encoding="utf-8")
        continue

    print(f"[{i+1}/{len(en_images)}] {img_path.name}...", end=" ")

    try:
        text = ocr_page_english(str(img_path), output_format=BEST_FORMAT)
        out_path.write_text(text, encoding="utf-8")
        results[img_path.stem] = text
        print(f"✅ {len(text)} chars")
    except Exception as e:
        print(f"❌ {e}")

    time.sleep(1)  # small delay to be polite to the API

print(f"\n✅ Done — {len(results)} pages processed")

## Step 6 — Side-by-Side Quality Check

Display the original image next to the extracted text for a direct comparison.

In [ ]:
from IPython.display import display, Image as IPImage

PAGE_TO_CHECK = 1  # change this to inspect different pages

md_files = sorted(OUTPUT_DIR.glob("*.md"))
if len(md_files) >= PAGE_TO_CHECK:
    img_file   = en_images[PAGE_TO_CHECK - 1]
    text_file  = md_files[PAGE_TO_CHECK - 1]

    print(f"=== ORIGINAL IMAGE: {img_file.name} ===")
    display(IPImage(filename=str(img_file), width=600))

    print(f"\n=== EXTRACTED TEXT: {text_file.name} ===")
    print(text_file.read_text(encoding="utf-8"))

## Step 7 — Spike Summary

In [ ]:
md_files = sorted(OUTPUT_DIR.glob("*.md"))
total_chars = sum(f.read_text(encoding="utf-8").__len__() for f in md_files)

print("=" * 50)
print("SPIKE 03 — RESULTS")
print("=" * 50)
print(f"Pages processed     : {len(md_files)} / {len(en_images)}")
print(f"Total chars extracted: {total_chars:,}")
print(f"Avg chars per page  : {total_chars // max(len(md_files),1):,}")
print()

# Check table preservation
tables_found = sum(1 for f in md_files if '|' in f.read_text(encoding='utf-8'))
print(f"Pages with Markdown tables: {tables_found}")
print()

if len(md_files) == len(en_images) and total_chars > 1000:
    print("✅ SPIKE PASSED — English OCR output ready for Spike 04 (PageIndex)")
    print(f"   Files in: {OUTPUT_DIR.resolve()}")
else:
    print("⚠️  SPIKE INCOMPLETE")